# ComplaintRouter — Download CFPB Complaint Data

On **August 14, 2026** the CFPB stopped publishing consumer complaint narratives. The current
bulk file at `https://files.consumerfinance.gov/ccdb/complaints.csv.zip` and the complaint API
no longer contain the `Consumer complaint narrative` field, which is the input for every model
in this project.

This notebook therefore downloads the **official CFPB bulk file as archived by the Internet
Archive on August 15, 2026**, the last capture that still includes narratives. Its 16 columns
match the CFPB exports the team originally used. The file is saved to `data/raw/`
(gitignored) and is read directly from the zip by `01_data_cleaning.ipynb`.

In [1]:
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from tqdm import tqdm

In [2]:
ORIGINAL_URL = "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"
SNAPSHOT = "20260815141332"

# The "id_" flag asks the Wayback Machine for the original bytes, without any rewriting
DATA_URL = f"https://web.archive.org/web/{SNAPSHOT}id_/{ORIGINAL_URL}"
EXPECTED_BYTES = 1_415_011_387

raw_dir = Path("../data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)

zip_path = raw_dir / "complaints_2026-08-15_archive.csv.zip"
info_path = raw_dir / "DOWNLOAD_INFO.txt"

## 1. Download the Archived Snapshot

Downloads from the Internet Archive can be slow or interrupted, so partial downloads are
resumed with HTTP range requests. The download is skipped when a complete copy already exists.

In [3]:
def download(url, path, expected_bytes, max_attempts=10):
    part_path = path.with_name(path.name + ".part")

    for attempt in range(1, max_attempts + 1):
        done = part_path.stat().st_size if part_path.exists() else 0
        if done >= expected_bytes:
            break

        headers = {"Range": f"bytes={done}-"} if done else {}
        try:
            with requests.get(url, headers=headers, stream=True, timeout=120) as response:
                response.raise_for_status()
                if done and response.status_code != 206:
                    done = 0  # server ignored the range request, so start over

                with open(part_path, "ab" if done else "wb") as f, tqdm(
                    total=expected_bytes, initial=done, unit="B", unit_scale=True,
                    desc=f"attempt {attempt}", mininterval=30,
                ) as progress:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
                        progress.update(len(chunk))
        except (requests.RequestException, OSError) as error:
            print(f"Attempt {attempt} interrupted: {error}")

    if part_path.stat().st_size != expected_bytes:
        raise RuntimeError(f"Incomplete download: {part_path.stat().st_size:,} of {expected_bytes:,} bytes")

    part_path.rename(path)


if zip_path.exists() and zip_path.stat().st_size == EXPECTED_BYTES:
    print(f"Using existing file: {zip_path} ({zip_path.stat().st_size:,} bytes)")
else:
    download(DATA_URL, zip_path, EXPECTED_BYTES)

    info_path.write_text(
        f"source: {DATA_URL}\n"
        f"original_url: {ORIGINAL_URL}\n"
        f"archived_utc: {SNAPSHOT}\n"
        f"bytes: {zip_path.stat().st_size}\n"
        f"downloaded_utc: {datetime.now(timezone.utc).isoformat(timespec='seconds')}\n"
    )

    print(f"Saved {zip_path.stat().st_size:,} bytes to {zip_path}")

Using existing file: ..\data\raw\complaints_2026-08-15_archive.csv.zip (1,415,011,387 bytes)


## 2. Verify the Archive

In [4]:
print(info_path.read_text())

with zipfile.ZipFile(zip_path) as archive:
    for member in archive.infolist():
        print(f"{member.filename}: {member.file_size / 1e9:,.2f} GB uncompressed")

    bad_member = archive.testzip()

print("Archive OK" if bad_member is None else f"Corrupt member: {bad_member}")

source: https://web.archive.org/web/20260815141332id_/https://files.consumerfinance.gov/ccdb/complaints.csv.zip
original_url: https://files.consumerfinance.gov/ccdb/complaints.csv.zip
archived_utc: 20260815141332
bytes: 1415011387
downloaded_utc: 2026-10-25T21:02:47+00:00

complaints.csv: 9.13 GB uncompressed


Archive OK


## 3. Confirm the Narrative Column Is Present

In [5]:
preview = pd.read_csv(zip_path, nrows=5)

print("Columns:")
print(preview.columns.tolist())
print()
print("Narrative column present:", "Consumer complaint narrative" in preview.columns)

Columns:
['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Complaint ID']

Narrative column present: True
